# Frequency Adaptation Test

This notebook tests the reusable basin-scale frequency adaptation workflow.

The workflow:

1. loads the baseline basin risk table
2. defines which basins are affected
3. defines an RP shift table
4. builds basin-scale frequency scenario curves
5. runs the simulation and compares results


In [9]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd

from sovereign.flood import (
    BasinLossCurve,
    apply_basin_frequency_shift,
    build_basin_curves,
    build_frequency_scenario_curves,
    build_uniform_frequency_shift_table,
    extract_sectoral_losses,
    run_simulation,
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
# USER CONFIG
model = "wri"
n_years = 5000

# Frequency settings
shift_factor = 1.50
degrade_protection = True
n_target_basins = 10

root = Path.cwd().parent
risk_basin_path = root / "outputs" / "flood" / "risk" / "basins" / f"risk_basins_m-{model}.csv"
copula_path = root / "outputs" / "flood" / "dependence" / "copulas" / "copula_random_numbers.gzip"


In [11]:
# Load baseline data
risk_data = pd.read_csv(risk_basin_path)
risk_data = risk_data.iloc[:, 1:]
risk_data["AEP"] = 1 / risk_data["RP"]
risk_data["Pr_L_AEP"] = np.where(risk_data["Pr_L"] == 0, 0, 1 / risk_data["Pr_L"])
risk_data.reset_index(drop=True, inplace=True)

copula_random_numbers = pd.read_parquet(copula_path).iloc[:n_years].copy()

risk_data.head()


,FID,GID_1,NAME,HB_L6,Pr_L,damages,adapted_damages,RP,Sector,AEP,Pr_L_AEP
0,0,UGA.3_1,Arua,1.061054e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
1,1,UGA.47_1,Nebbi,1.061054e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
2,2,UGA.27_1,Kitgum,1.060999e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
3,3,UGA.41_1,Moyo,1.061033e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
4,4,UGA.3_1,Arua,1.061033e+09,2.0,6160.324707,6160.324707,5,Public,0.2,0.5


In [12]:
# Build baseline curves
baseline_curves: dict[int, BasinLossCurve] = build_basin_curves(risk_data)

# Example target basins: first N unique basins from the risk table
target_basins = sorted(risk_data["HB_L6"].unique())[:n_target_basins]
target_basins[:10]


[1060999120.0,
 1061016240.0,
 1061022540.0,
 1061029030.0,
 1061033480.0,
 1061033490.0,
 1061041420.0,
 1061041490.0,
 1061051360.0,
 1061051510.0]

## Build Frequency Shift Inputs

For quick testing we use one multiplicative factor applied to all return
periods. Later this can be replaced with a custom RP-to-RP or AEP-to-AEP table.


In [13]:
frequency_shift_df = build_uniform_frequency_shift_table(
    return_periods=risk_data["RP"].unique(),
    shift_factor=shift_factor,
)
frequency_shift_df


,RP,RP_future
0,5.0,7.5
1,10.0,15.0
2,25.0,37.5
3,50.0,75.0
4,100.0,150.0
5,250.0,375.0
6,500.0,750.0
7,1000.0,1500.0


In [14]:
# Optional inspection of the shifted basin risk rows
shifted_risk_df = apply_basin_frequency_shift(
    risk_df=risk_data,
    frequency_shift_df=frequency_shift_df,
    basin_ids=target_basins,
    degrade_protection=degrade_protection,
)
shifted_risk_df.head()


,FID,GID_1,NAME,HB_L6,Pr_L,damages,adapted_damages,RP,Sector,AEP,Pr_L_AEP,component_type
0,0,UGA.3_1,Arua,1.061054e+09,2.0,0.000000,0.000000,5.0,Public,0.200000,0.500000,frequency_shifted
1,1,UGA.47_1,Nebbi,1.061054e+09,2.0,0.000000,0.000000,5.0,Public,0.200000,0.500000,frequency_shifted
2,2,UGA.27_1,Kitgum,1.060999e+09,7.5,0.000000,0.000000,7.5,Public,0.133333,0.133333,frequency_shifted
3,3,UGA.41_1,Moyo,1.061033e+09,7.5,0.000000,0.000000,7.5,Public,0.133333,0.133333,frequency_shifted
4,4,UGA.3_1,Arua,1.061033e+09,7.5,6160.324707,6160.324707,7.5,Public,0.133333,0.133333,frequency_shifted


## Build Scenario Curves And Run Simulation


In [15]:
frequency_scenario_curves = build_frequency_scenario_curves(
    baseline_risk_df=risk_data,
    frequency_shift_df=frequency_shift_df,
    basin_ids=target_basins,
    degrade_protection=degrade_protection,
)

baseline_losses, frequency_adapted_losses = run_simulation(
    baseline_curves,
    frequency_scenario_curves,
    n_years,
    copula_random_numbers,
)


100%|█████████████████████████████████████████████████████████████████████████████| 5000/5000 [00:43<00:00, 115.10it/s]


In [16]:
baseline_sectoral_loss = extract_sectoral_losses(baseline_losses, n_years)
frequency_sectoral_loss = extract_sectoral_losses(frequency_adapted_losses, n_years)

comparison_df = pd.DataFrame({
    "metric": ["GVA_loss", "CAP_dam", "AGR_loss", "MAN_loss", "SER_loss", "PUB_dam", "PRI_dam"],
    "baseline_aal": [baseline_sectoral_loss[c].mean() for c in ["GVA_loss", "CAP_dam", "AGR_loss", "MAN_loss", "SER_loss", "PUB_dam", "PRI_dam"]],
    "frequency_aal": [frequency_sectoral_loss[c].mean() for c in ["GVA_loss", "CAP_dam", "AGR_loss", "MAN_loss", "SER_loss", "PUB_dam", "PRI_dam"]],
})
comparison_df["aal_change"] = comparison_df["frequency_aal"] - comparison_df["baseline_aal"]
comparison_df["pct_change"] = np.where(
    comparison_df["baseline_aal"] != 0,
    100 * comparison_df["aal_change"] / comparison_df["baseline_aal"],
    np.nan,
)
comparison_df


,metric,baseline_aal,frequency_aal,aal_change,pct_change
0,GVA_loss,4.626507e+07,4.586687e+07,-398194.522116,-0.860681
1,CAP_dam,5.616184e+07,5.553145e+07,-630389.622751,-1.122452
2,AGR_loss,1.494232e+07,1.455348e+07,-388839.432209,-2.602269
3,MAN_loss,1.028712e+07,1.028420e+07,-2925.310585,-0.028437
4,SER_loss,2.103563e+07,2.102920e+07,-6429.779322,-0.030566
5,PUB_dam,2.086799e+07,2.066580e+07,-202191.663061,-0.968908
6,PRI_dam,3.529385e+07,3.486565e+07,-428197.959690,-1.213237


## Notes

To customize this workflow, change:

- `target_basins`
- `shift_factor`
- `degrade_protection`

If you want a non-uniform shift, replace `build_uniform_frequency_shift_table(...)`
with your own dataframe containing either:

- `RP` and `RP_future`
- or `AEP` and `AEP_future`
